In [1]:
"""
Interactive Graph POC using Plotly
-----------------------------------
Run:  pip install plotly
Then: python interactive_graph.py
 
Opens a self-contained HTML file in your browser with:
  - Hover tooltips showing precise (x, y) values
  - Zoom / pan
  - Click-to-toggle series visibility
  - Download as PNG button (built-in)
"""
 
import math
import plotly.graph_objects as go
 
# ── Sample data ───────────────────────────────────────────────────────────────
x  = [i * 0.1 for i in range(100)]          # 0.0 → 9.9
y1 = [math.sin(v) for v in x]               # sine wave
y2 = [math.sin(v) * math.exp(-v / 6) for v in x]  # damped sine
 
# ── Build the figure ──────────────────────────────────────────────────────────
fig = go.Figure()
 
fig.add_trace(go.Scatter(
    x=x, y=y1,
    mode="lines",
    name="sin(x)",
    line=dict(color="#4f8ef7", width=2.5),
    hovertemplate="x: %{x:.2f}<br>y: %{y:.4f}<extra></extra>",
))
 
fig.add_trace(go.Scatter(
    x=x, y=y2,
    mode="lines",
    name="damped sin(x)",
    line=dict(color="#f76f4f", width=2.5, dash="dot"),
    hovertemplate="x: %{x:.2f}<br>y: %{y:.4f}<extra></extra>",
))
 
# ── Layout / styling ──────────────────────────────────────────────────────────
fig.update_layout(
    title=dict(text="Interactive Graph POC", font=dict(size=22)),
    xaxis=dict(title="x", showgrid=True, gridcolor="#e5e5e5"),
    yaxis=dict(title="y", showgrid=True, gridcolor="#e5e5e5"),
    hovermode="x unified",   # shows all series values at cursor x position
    legend=dict(
        orientation="h",
        yanchor="bottom", y=1.02,
        xanchor="right",  x=1,
    ),
    plot_bgcolor="white",
    paper_bgcolor="white",
    font=dict(family="Georgia, serif", size=14),
    margin=dict(l=60, r=30, t=80, b=60),
)
 
# ── Export ────────────────────────────────────────────────────────────────────
output_file = "interactive_graph.html"
fig.write_html(
    output_file,
    include_plotlyjs="cdn",   # ~3 kB stub; full Plotly loaded from CDN
    full_html=True,
)
 
print(f"✅  Saved → {output_file}")
print("Open it in any browser — no server needed.")
 
# Uncomment to auto-open in your default browser:
# fig.show()

✅  Saved → interactive_graph.html
Open it in any browser — no server needed.


In [2]:
"""
Economist-style hover highlight chart
--------------------------------------
Install:  pip install plotly
Run:      python hover_highlight.py
Opens:    hover_highlight.html  (or auto-opens in browser)
 
Hover over any gray line → it highlights red.
Named series stay in their own colors throughout.
"""
 
import math
import random
import plotly.graph_objects as go
 
random.seed(42)
 
# ── Named series (the ones that matter) ───────────────────────────────────────
years = [y / 2 for y in range(2017 * 2, 2026 * 2 + 1)]  # 2017.0 … 2026.0 step 0.5
 
named_series = [
    {
        "label": "Economy",
        "color": "#e3120b",
        "values": [
            19,20,18,17,16,15,16,22,28,30,32,35,35,32,30,27,25,22,21,
        ],
    },
    {
        "label": "Immigration",
        "color": "#f6423c",
        "values": [
            10,10,9,12,13,14,10,9,10,11,12,13,12,14,14,14,11,11,11,
        ],
    },
    {
        "label": "Health care",
        "color": "#FCB6A9",
        "values": [
            17,17,16,14,14,13,12,12,11,11,11,10,10,11,11,11,12,12,12,
        ],
    },
    {
        "label": "Civil rights",
        "color": "#666666",
        "values": [
            8,8,9,9,8,8,11,13,11,10,9,9,9,9,9,9,9,9,9,
        ],
    },
    {
        "label": "National security",
        "color": "#333333",
        "values": [
            11,11,11,11,10,10,10,10,10,10,10,10,10,10,10,10,10,10,10,
        ],
    },
]
 
# Pad / trim each series to match the years list length
n = len(years)
for s in named_series:
    v = s["values"]
    if len(v) < n:
        v += [v[-1]] * (n - len(v))
    s["values"] = v[:n]
 
 
# ── Background gray lines (simulated other issues) ─────────────────────────────
def make_bg_line(seed_offset):
    amp = 4 + random.random() * 8
    shift = random.random() * 3
    return [
        round(3 + amp * abs(math.sin((y - shift) * 1.3 + seed_offset)))
        for y in years
    ]
 
 
NUM_BG = 22
bg_lines = [make_bg_line(i) for i in range(NUM_BG)]
 
 
# ── Build the figure ───────────────────────────────────────────────────────────
fig = go.Figure()
 
# 1. Background lines — gray by default, red on hover
#    Plotly doesn't have a "highlight one on hover" mode built-in,
#    but we can get very close: make each line interactive with a
#    visible hover tooltip, and style them so they stand out on hover
#    using the `line.color` + custom hovertemplate trick.
#
#    For true "turn red on hover" we use updatemenus / JS via
#    write_html() with post_script — the cleanest pure-Python approach.
 
for i, vals in enumerate(bg_lines):
    fig.add_trace(
        go.Scatter(
            x=years,
            y=vals,
            mode="lines",
            name=f"_bg{i}",
            showlegend=False,
            line=dict(color="rgba(160,160,160,0.25)", width=1.2),
            hovertemplate="<b>%{y}%</b> — %{x:.1f}<extra></extra>",
            # Each bg trace gets a unique CSS class via uid for JS targeting
            uid=f"bg{i}",
        )
    )
 
# 2. Named series on top
for s in named_series:
    fig.add_trace(
        go.Scatter(
            x=years,
            y=s["values"],
            mode="lines",
            name=s["label"],
            line=dict(color=s["color"], width=2.5),
            hovertemplate=f"<b>{s['label']}</b>: %{{y}}%<extra></extra>",
        )
    )
 
# ── Layout ─────────────────────────────────────────────────────────────────────
fig.update_layout(
    title=dict(
        text="Most important issues, 2017–2026",
        font=dict(size=18, family="Georgia, serif"),
        x=0,
        xanchor="left",
    ),
    xaxis=dict(
        title="",
        tickvals=list(range(2017, 2027)),
        ticktext=[str(y) for y in range(2017, 2027)],
        showgrid=True,
        gridcolor="#ebebeb",
        zeroline=False,
    ),
    yaxis=dict(
        title="% naming as most important issue",
        range=[0, 42],
        ticksuffix="%",
        showgrid=True,
        gridcolor="#ebebeb",
        zeroline=True,
        zerolinecolor="#ccc",
    ),
    hovermode="x unified",
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="left",
        x=0,
        font=dict(size=12),
    ),
    plot_bgcolor="white",
    paper_bgcolor="white",
    font=dict(family="Georgia, serif", size=13),
    margin=dict(l=60, r=20, t=80, b=60),
    height=500,
)
 
# ── Inject JavaScript for the red-on-hover effect ─────────────────────────────
#    post_script runs after Plotly renders, giving us access to the div + Plotly.
#    We listen for plotly_hover events and restyle the hovered bg trace red.
 
js = """
var gd = document.getElementById('{plot_id}');
var NUM_BG = """ + str(NUM_BG) + """;
 
gd.on('plotly_hover', function(data) {
    var pt = data.points[0];
    var traceIdx = pt.curveNumber;
 
    // Only react if it's a background trace
    if (traceIdx >= NUM_BG) return;
 
    // Reset all bg traces to gray
    var grayUpdate = { 'line.color': [], 'line.width': [] };
    for (var i = 0; i < NUM_BG; i++) {
        grayUpdate['line.color'].push('rgba(160,160,160,0.25)');
        grayUpdate['line.width'].push(1.2);
    }
    Plotly.restyle(gd, grayUpdate, [...Array(NUM_BG).keys()]);
 
    // Highlight the hovered trace red
    Plotly.restyle(gd, {
        'line.color': '#e3120b',
        'line.width': 2.5
    }, [traceIdx]);
});
 
gd.on('plotly_unhover', function() {
    // Reset all bg traces to gray on mouse-out
    var grayUpdate = { 'line.color': [], 'line.width': [] };
    for (var i = 0; i < NUM_BG; i++) {
        grayUpdate['line.color'].push('rgba(160,160,160,0.25)');
        grayUpdate['line.width'].push(1.2);
    }
    Plotly.restyle(gd, grayUpdate, [...Array(NUM_BG).keys()]);
});
"""
 
output_file = "hover_highlight.html"
fig.write_html(
    output_file,
    include_plotlyjs="cdn",
    post_script=js,          # <-- this is the key: inject JS after render
    full_html=True,
)
 
print(f"✅  Saved → {output_file}")
print("   Open in any browser — no server needed.")
 
# Uncomment to auto-open:
fig.show()

✅  Saved → hover_highlight.html
   Open in any browser — no server needed.
